<a href="https://colab.research.google.com/github/guillaumevalette2-hash/mse_gh/blob/main/multi_greedy_porow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from itertools import product
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
import time

params = {
    "n_ambiant": 4, "deg_P": 3, "n_terms_poly": 20,
   "seeds": {1: 3253112123, 2: 8222535114, 3: 5575225122373,
              42:5941231338, 43:6885511236, 44: 4721512234, 11:17233381, 12: 2353213671},
    "weights":   {0: 0, 1: 1, 2: 0.1, 3: 0.01},
    "n_train":   50,
    "n_test":    5000,
    "n_levels":  4,       # nb de fonctions f_i cylindriques
    "n_levels_wnd": 4,    # nb de fonctions f_i Wendland anisotropes (support compact ->
                          # experts plus indépendants ; régularité C^{2k} minimale telle
                          # que 2k >= ordre max des poids Sobolev)
    "k_loss":    3.0,     # sélection finale : garder les experts avec loss <= k_loss * meilleure loss
    "deg_poly_expert": -1,      # degré de l'expert polynomial (dérivées analytiques exactes)
                               # -1 -> expert polynomial désactivé
    "n_dict":    5000,     # taille dictionnaire par niveau
    "n_centres": 800,      # centres retenus par niveau
    "sigma_min": 0.1,      # borne inférieure des sigma_i
    "sigma_max": 3.0,      # borne supérieure des sigma_i
    "lambda_reg": 1e-5,    # UNIQUE régularisation Sobolev (Phase 1, sélection, Phase 2) :
                           # la norme Sobolev est indépendante de la base -> une seule loss
    "n_G":       2000,      # points pour G de chaque f_i
    "n_G_H":     5000,     # points pour la Gram finale sur H
    "n_levels_miso": 6,     # niveaux multi-isotropes : UN sigma aléatoire par centre,
                            # log-uniforme dans [sigma_min, sigma_max] (0 -> étape sautée)
    "miso_greedy": True,    # True  : les niveaux miso sont construits en cascade gloutonne
                            #         (chaque niveau fitte le RÉSIDU du précédent) et seul
                            #         le DERNIER (= la somme cumulée) est conservé comme expert.
                            # False : anciens niveaux miso indépendants (tous conservés).
}
params["n_unlabeled"]        = 4000 - params["n_train"]
params["train_center_ratio"] = 0.5 + params["n_train"] / (2 * 4000)

# ══════════════════════════════════════════════════════════════════════════════
# POLYNÔMES ET CLOUD
# ══════════════════════════════════════════════════════════════════════════════
def random_sparse_polynomial(d, degree, n_terms, seed=None):
    rng = np.random.default_rng(seed)
    all_indices = [exp for exp in product(range(degree+1), repeat=d)
                   if sum(exp) <= degree]
    indices = rng.choice(all_indices, size=n_terms, replace=False)
    coeffs  = rng.normal(size=n_terms)
    def P(x):
        y = np.zeros(x.shape[0])
        for c, alpha in zip(coeffs, indices):
            term = np.ones(x.shape[0])
            for j, e in enumerate(alpha):
                if e > 0: term *= x[:,j]**e
            y += c * term
        return y
    zero_val = sum(c for c, alpha in zip(coeffs, indices)
                   if all(e == 0 for e in alpha))
    def P_zero(x): return P(x) - zero_val
    return P_zero, indices, coeffs

def normalize_polynomial(P, indices, coeffs):
    max_c = np.max(np.abs(coeffs))
    if max_c > 0:
        nc = coeffs / max_c
        def Pn(x):
            y = np.zeros(x.shape[0])
            for c, alpha in zip(nc, indices):
                term = np.ones(x.shape[0])
                for j, e in enumerate(alpha):
                    if e > 0: term *= x[:,j]**e
                y += c * term
            return y
        return Pn
    return P

P1,i1,c1 = random_sparse_polynomial(params["n_ambiant"],params["deg_P"],params["n_terms_poly"],params["seeds"][1])
P2,i2,c2 = random_sparse_polynomial(params["n_ambiant"],params["deg_P"],params["n_terms_poly"],params["seeds"][2])
P3,i3,c3 = random_sparse_polynomial(params["n_ambiant"],params["deg_P"],params["n_terms_poly"],params["seeds"][3])
P1=normalize_polynomial(P1,i1,c1); P2=normalize_polynomial(P2,i2,c2); P3=normalize_polynomial(P3,i3,c3)
def Q(x): return P1(x)*P2(x)*P3(x)

def grad_Q_analytical(X):
    eps=1e-5; d=X.shape[1]; p1=P1(X); p2=P2(X); p3=P3(X)
    grad=np.zeros_like(X)
    for k in range(d):
        Xp=X.copy(); Xp[:,k]+=eps; Xm=X.copy(); Xm[:,k]-=eps
        dp1=(P1(Xp)-P1(Xm))/(2*eps); dp2=(P2(Xp)-P2(Xm))/(2*eps); dp3=(P3(Xp)-P3(Xm))/(2*eps)
        grad[:,k]=dp1*p2*p3+p1*dp2*p3+p1*p2*dp3
    return grad

def project_to_Q_zero(X_init, n_steps=150, lr=0.05, tol=1e-4):
    X=X_init.copy()
    for _ in range(n_steps):
        q=Q(X); gq=grad_Q_analytical(X); gq2=2*q[:,None]*gq
        norm=np.linalg.norm(gq2,axis=1,keepdims=True)+1e-12
        X=X-lr*gq2/norm; X=np.clip(X,0.,1.)
        if np.abs(Q(X)).max()<tol*0.1: break
    return X, np.abs(Q(X))<tol

def sample_on_Q_zero(n_target, d, seed=None):
    rng=np.random.default_rng(seed); collected=[]; n_col=0
    while n_col<n_target:
        n_batch=max((n_target-n_col)*4,200)
        Xi=rng.uniform(0,1,(int(n_batch),d)); Xp,conv=project_to_Q_zero(Xi)
        good=Xp[conv]
        if len(good)>0: collected.append(good); n_col+=len(good)
        print(f"  collectés:{n_col}/{n_target} ({conv.mean():.1%})",end='\r')
    print(); return np.vstack(collected)[:n_target]

print("Génération du cloud sur {Q=0}...")
d=params["n_ambiant"]
X_train     = sample_on_Q_zero(params["n_train"],    d, seed=params["seeds"][42])
X_test      = sample_on_Q_zero(params["n_test"],     d, seed=params["seeds"][43])
X_unlabeled = sample_on_Q_zero(params["n_unlabeled"],d, seed=params["seeds"][44])
print(f"  X_train:{X_train.shape}  X_unlabeled:{X_unlabeled.shape}")

P_target1, indicest1, coeffst1 = random_sparse_polynomial(params["n_ambiant"],
                                           4, params["n_terms_poly"], seed=params["seeds"][11])
P_target2, _, _ = random_sparse_polynomial(params["n_ambiant"],
                                           4, params["n_terms_poly"], seed=params["seeds"][12])
Ptarget1 = normalize_polynomial(P_target1, indicest1, coeffst1)
Ptarget2 = normalize_polynomial(P_target2, indicest1, coeffst1)

def target_function(X):
    return np.minimum(np.abs(P_target1(2*X)),1) +  np.minimum(np.abs(P_target2(2*X)),1)

y_train=target_function(X_train); y_test=target_function(X_test)
X_all=np.vstack([X_train,X_unlabeled]); N_all=len(X_all)

# ══════════════════════════════════════════════════════════════════════════════
# GAUSSIENNES CYLINDRIQUES
# ══════════════════════════════════════════════════════════════════════════════
def sample_candidates(X_train, X_unlabeled, n_candidates, train_ratio, rng):
    n_tr=min(int(round(n_candidates*train_ratio)),X_train.shape[0])
    n_ul=min(n_candidates-n_tr,X_unlabeled.shape[0]); parts=[]
    if n_tr>0: parts.append(X_train[rng.choice(X_train.shape[0],n_tr,replace=False)])
    if n_ul>0: parts.append(X_unlabeled[rng.choice(X_unlabeled.shape[0],n_ul,replace=False)])
    centers=np.vstack(parts)
    log_s=rng.uniform(np.log(params["sigma_min"]),np.log(params["sigma_max"]),
                      size=(len(centers),d))
    sigmas=np.exp(log_s)
    return centers, sigmas

def sample_candidates_miso(X_train, X_unlabeled, n_candidates, train_ratio, rng):
    """Multi-isotrope (ancienne famille iso) : gaussiennes isotropes avec UN sigma
    aléatoire par centre, log-uniforme dans [sigma_min, sigma_max], répété sur
    les d dimensions."""
    n_tr=min(int(round(n_candidates*train_ratio)),X_train.shape[0])
    n_ul=min(n_candidates-n_tr,X_unlabeled.shape[0]); parts=[]
    if n_tr>0: parts.append(X_train[rng.choice(X_train.shape[0],n_tr,replace=False)])
    if n_ul>0: parts.append(X_unlabeled[rng.choice(X_unlabeled.shape[0],n_ul,replace=False)])
    centers=np.vstack(parts)
    log_s=rng.uniform(np.log(params["sigma_min"]),np.log(params["sigma_max"]),
                      size=len(centers))                      # UN sigma par centre
    sigmas=np.tile(np.exp(log_s)[:,None],(1,d))               # répété sur les d dims
    return centers, sigmas

def cyl_features(X, centers, sigmas):
    diff=X[:,None,:]-centers[None,:,:]
    sq_w=np.sum(diff**2/sigmas[None,:,:]**2, axis=2)
    return np.exp(-sq_w)

def cyl_derivatives(X, centers, sigmas, weights=None):
    d_=X.shape[1]; n=X.shape[0]; m=centers.shape[0]
    diff=X[:,None,:]-centers[None,:,:]
    inv_s2=1./sigmas**2
    diff_w=diff*inv_s2[None,:,:]
    sq_w=np.sum(diff*diff_w, axis=2)
    phi=np.exp(-sq_w)
    grad=-2*diff_w*phi[:,:,None]
    need_hess  = weights is None or weights.get(2,0.)!=0 or weights.get(3,0.)!=0
    need_third = weights is None or weights.get(3,0.)!=0
    if need_hess:
        Id=np.eye(d_)
        diag=-2*inv_s2[None,:,:,None]*Id[None,None,:,:]
        cross=4*np.einsum('nmi,nmj->nmij',diff_w,diff_w)
        hess=(diag+cross)*phi[:,:,None,None]
    else: hess=None
    if need_third:
        Id=np.eye(d_)
        inv_s2_bc=inv_s2[None,:,:]
        cubic=-8*np.einsum('nmi,nmj,nmk->nmijk',diff_w,diff_w,diff_w)
        sym=4*(np.einsum('ij,nmi,nmk->nmijk',Id,inv_s2_bc,diff_w)+
               np.einsum('ik,nmi,nmj->nmijk',Id,inv_s2_bc,diff_w)+
               np.einsum('jk,nmj,nmi->nmijk',Id,inv_s2_bc,diff_w))
        third=(cubic+sym)*phi[:,:,None,None,None]
    else: third=None
    return phi, grad, hess, third

# ── Wendland anisotropes ──────────────────────────────────────────────────────
# f(x) = φ(r),  r² = Σ_k (x_k - c_k)²/σ_k²,  support compact {r < 1}.
# Profil C^{2k} minimal tel que 2k >= ordre max des poids (poids ordre 3 -> C⁴).
# Dérivées cartésiennes via les facteurs radiaux h1=φ'/r, h2=h1'/r, h3=h2'/r
# (formes fermées ; pour C⁴ seul h3 demande une division protégée en r=0, et le
#  terme h3·u_a u_b u_c ~ r² y tend vers 0, donc la limite est correcte).
def _wnd_profile(max_order):
    k = max(1, int(np.ceil(max_order/2)))
    sd = lambda num, r: np.where(r > 1e-12, num/np.maximum(r, 1e-12), 0.0)
    if k == 1:      # C² : suffit si ordre max <= 2
        return (lambda r,rp: rp**4*(4*r+1),
                lambda r,rp: -20*rp**3,
                lambda r,rp: sd(60*rp**2, r),
                None)                                   # ordre 3 non défini en C²
    elif k == 2:    # C⁴ : suffit si ordre max <= 4
        return (lambda r,rp: rp**6*(35*r**2+18*r+3)/3,
                lambda r,rp: -(56/3)*rp**5*(5*r+1),
                lambda r,rp: 560.*rp**4,
                lambda r,rp: sd(-2240.*rp**3, r))
    else:           # C⁶
        return (lambda r,rp: rp**8*(32*r**3+25*r**2+8*r+1),
                lambda r,rp: -22*rp**7*(16*r**2+7*r+1),
                lambda r,rp: 528*rp**6*(6*r+1),
                lambda r,rp: -22176.*rp**5)

_wnd_max_order = max([o for o, w in params["weights"].items() if w != 0])
WND_PHI, WND_H1, WND_H2, WND_H3 = _wnd_profile(_wnd_max_order)

def wnd_features(X, centers, sigmas):
    u = (X[:,None,:]-centers[None,:,:])/sigmas[None,:,:]
    r = np.sqrt(np.sum(u**2, axis=2)); rp = np.maximum(1.-r, 0.)
    return WND_PHI(r, rp)

def wnd_derivatives(X, centers, sigmas, weights=None):
    d_ = X.shape[1]
    u  = (X[:,None,:]-centers[None,:,:])/sigmas[None,:,:]   # (n,m,d)
    r  = np.sqrt(np.sum(u**2, axis=2)); rp = np.maximum(1.-r, 0.)
    inv_s  = 1./sigmas                    # (m,d)
    inv_s2 = inv_s**2
    us  = u*inv_s[None,:,:]               # u_a/σ_a
    phi = WND_PHI(r, rp)
    H1  = WND_H1(r, rp)
    grad = H1[:,:,None]*us                # grad_a = h1 · u_a/σ_a
    need_hess  = weights is None or weights.get(2,0.)!=0 or weights.get(3,0.)!=0
    need_third = weights is None or weights.get(3,0.)!=0
    if need_hess:
        Id = np.eye(d_)
        H2 = WND_H2(r, rp)
        # hess_ab = h2 · (u_a/σ_a)(u_b/σ_b) + h1 · δ_ab/σ_a²
        hess = (H2[:,:,None,None]*np.einsum('nmi,nmj->nmij', us, us)
                + H1[:,:,None,None]*(Id[None,None,:,:]*inv_s2[None,:,:,None]))
    else: hess=None
    if need_third:
        assert WND_H3 is not None, "poids d'ordre 3 : profil Wendland C⁴ minimum requis"
        Id = np.eye(d_)
        H3 = WND_H3(r, rp)
        # third_abc = h3·us_a us_b us_c
        #           + h2·(δ_ab us_c/σ_a² + δ_ac us_b/σ_a² + δ_bc us_a/σ_b²)
        cubic = H3[:,:,None,None,None]*np.einsum('nmi,nmj,nmk->nmijk', us, us, us)
        sym = (np.einsum('ij,mi,nmk->nmijk', Id, inv_s2, us)
              +np.einsum('ik,mi,nmj->nmijk', Id, inv_s2, us)
              +np.einsum('jk,mj,nmi->nmijk', Id, inv_s2, us))
        third = cubic + WND_H2(r, rp)[:,:,None,None,None]*sym
    else: third=None
    return phi, grad, hess, third

def build_G(phi, grad, hess, third, weights, n):
    G=np.zeros((phi.shape[1],phi.shape[1]))
    w0=weights.get(0,0.)
    if w0: G+=w0*(phi.T@phi)/n
    w1=weights.get(1,0.)
    if w1 and grad  is not None: G+=w1*np.einsum('xik,xjk->ij',    grad, grad, optimize='optimal')/n
    w2=weights.get(2,0.)
    if w2 and hess  is not None: G+=w2*np.einsum('xikl,xjkl->ij',  hess, hess, optimize='optimal')/n
    w3=weights.get(3,0.)
    if w3 and third is not None: G+=w3*np.einsum('xiklm,xjklm->ij',third,third,optimize='optimal')/n
    return G

def sobolev_norm2(derivs, weights, idx=None):
    """||f||²_H = Σ_o w_o · mean_x ||D^o f(x)||²_F  pour une fonction déjà
    évaluée (dict phi/grad/hess/third de shapes (n,), (n,d), (n,d,d), (n,d,d,d)).
    idx : sous-échantillon optionnel de points."""
    tot = 0.
    for key, order, axes in (('phi',0,()), ('grad',1,(1,)),
                             ('hess',2,(1,2)), ('third',3,(1,2,3))):
        w = weights.get(order, 0.)
        if not w:            continue
        T = derivs.get(key)
        if T is None:        continue
        if idx is not None:  T = T[idx]
        tot += w * (np.mean(T**2) if order == 0
                    else np.mean(np.sum(T**2, axis=axes)))
    return tot

# ══════════════════════════════════════════════════════════════════════════════
# PHASE 1 : calculer n_levels fonctions f_i indépendantes
# Chaque f_i = argmin sur son propre dictionnaire cylindrique aléatoire
# ══════════════════════════════════════════════════════════════════════════════
MISO_GREEDY = bool(params.get("miso_greedy", False)) and params["n_levels_miso"] > 0

level_specs = ([("cyl",  i) for i in range(params["n_levels"])] +
               [("miso", i) for i in range(params["n_levels_miso"])] +
               [("wnd",  i) for i in range(params["n_levels_wnd"])])

# En mode glouton, les n_levels_miso niveaux miso ne produisent qu'UN SEUL expert
# (le dernier étage de la cascade = somme cumulée des résidus).
n_experts_total = len(level_specs) - (params["n_levels_miso"] - 1 if MISO_GREEDY else 0)
n_levels_total  = n_experts_total

expert_names = []
for typ, i in level_specs:
    if typ == "miso" and MISO_GREEDY:
        if i == params["n_levels_miso"] - 1:
            expert_names.append(f"misoG{params['n_levels_miso']}")
        continue
    expert_names.append(f"{typ}_{i+1:02d}")

print(f"\nPhase 1 : calcul de {n_experts_total} experts "
      f"({params['n_levels']} cyl + "
      f"{params['n_levels_miso']} miso"
      f"{' [glouton -> 1 expert]' if MISO_GREEDY else ''} + "
      f"{params['n_levels_wnd']} wnd C{2*max(1,int(np.ceil(_wnd_max_order/2)))})...")

F_train = np.zeros((len(X_train), n_experts_total))   # f_i(X_train)
F_test  = np.zeros((len(X_test),  n_experts_total))   # f_i(X_test)
F_all   = np.zeros((N_all,        n_experts_total))   # f_i(X_all)  pour Gram H

# Pour la Gram finale on a besoin des dérivées de chaque f_i sur n_G_H points
# On les stocke compressées : delta_phi/grad/hess/third = fi_derivs sur X_all
fi_derivs_list = []  # liste de dicts par niveau

times_p1 = []

# ── état de la cascade gloutonne miso (accumulateurs) ─────────────────────────
gr_res     = y_train.copy()     # résidu courant à fitter
gr_train   = np.zeros(len(X_train))
gr_test    = np.zeros(len(X_test))
gr_all     = np.zeros(N_all)
gr_derivs  = None               # dict cumulé des dérivées de la somme
gr_h2_hist = []                 # historique des ||f_cum||²_H par étage

slot = 0                        # colonne courante dans F_* / expert_names
for level, (lev_type, lev_idx) in enumerate(level_specs):
    t0 = time.time()
    greedy_step = (lev_type == "miso") and MISO_GREEDY
    last_greedy = greedy_step and (lev_idx == params["n_levels_miso"] - 1)
    # cible du fit : résidu courant en mode glouton, y_train sinon
    y_fit = gr_res if greedy_step else y_train

    # graines distinctes par famille pour éviter les mêmes centres
    rng = np.random.default_rng(
        seed={"cyl":0, "wnd":2000, "miso":3000}[lev_type]+lev_idx)

    # wnd : même échantillonnage anisotrope que cyl, seul le profil radial change
    feat_fn  = wnd_features    if lev_type=="wnd" else cyl_features
    deriv_fn = wnd_derivatives if lev_type=="wnd" else cyl_derivatives
    if lev_type == "miso":      # multi-isotrope : un sigma aléatoire par centre
        candidates, sigmas_cand = sample_candidates_miso(
            X_train, X_unlabeled, params["n_dict"], params["train_center_ratio"], rng)
    else:                       # cyl / wnd : anisotrope
        candidates, sigmas_cand = sample_candidates(
            X_train, X_unlabeled, params["n_dict"], params["train_center_ratio"], rng)

    # Score par corrélation avec la cible courante (résidu si glouton)
    A_cand = feat_fn(X_train, candidates, sigmas_cand)
    corr   = (A_cand.T @ y_fit) / len(X_train)
    scores = corr**2

    k     = min(params["n_centres"], len(candidates))
    top_k = np.argsort(scores)[-k:]
    centers_sel = candidates[top_k]
    sigmas_sel  = sigmas_cand[top_k]

    A     = A_cand[:, top_k]
    Atest = feat_fn(X_test, centers_sel, sigmas_sel)
    Aall  = feat_fn(X_all,  centers_sel, sigmas_sel)

    phi, grad, hess, third = deriv_fn(
        X_all, centers_sel, sigmas_sel, params["weights"])

    n_G   = min(N_all, params["n_G"])
    idx_G = rng.choice(N_all, size=n_G, replace=False)
    G = build_G(phi[idx_G],
                grad[idx_G]  if grad  is not None else None,
                hess[idx_G]  if hess  is not None else None,
                third[idx_G] if third is not None else None,
                params["weights"], n_G)

    lambda_reg = params["lambda_reg"]
    n = A.shape[0]
    M   = (A.T@A)/n + lambda_reg*G
    rhs = (A.T@y_fit)/n

    eigvals,eigvecs = np.linalg.eigh(M)
    thresh = max(lambda_reg, 0); mask = eigvals > thresh
    V=eigvecs[:,mask]; S=eigvals[mask]; coeffs=V@((V.T@rhs)/S)

    f_tr = A     @ coeffs
    f_te = Atest @ coeffs
    f_al = Aall  @ coeffs

    # Dérivées de l'étage courant sur X_all
    step_derivs = {
        'phi':   phi   @ coeffs,
        'grad':  np.einsum('xjk,j->xk',   grad, coeffs) if grad  is not None else None,
        'hess':  np.einsum('xjkl,j->xkl', hess, coeffs) if hess  is not None else None,
        'third': np.einsum('xjklm,j->xklm',third,coeffs) if third is not None else None,
    }

    loss_data  = np.mean((y_fit - f_tr)**2)
    loss_reg   = lambda_reg * coeffs @ G @ coeffs
    loss_total = loss_data + loss_reg
    t1=time.time(); times_p1.append(t1-t0)

    if greedy_step:
        # ── cascade : on accumule la somme et on met à jour le résidu ─────────
        gr_train += f_tr; gr_test += f_te; gr_all += f_al
        gr_res    = y_train - gr_train
        if gr_derivs is None:
            gr_derivs = step_derivs
        else:
            for key in ('phi','grad','hess','third'):
                if step_derivs[key] is None or gr_derivs[key] is None:
                    gr_derivs[key] = None
                else:
                    gr_derivs[key] = gr_derivs[key] + step_derivs[key]

        mse_cum = mean_squared_error(y_test, gr_test)
        # ── normes Sobolev : de l'étage seul, et de la cumulée ────────────────
        h2_step = sobolev_norm2(step_derivs, params["weights"])
        h2_cum  = sobolev_norm2(gr_derivs,   params["weights"])
        gr_h2_hist.append(h2_cum)
        dh2 = h2_cum - (gr_h2_hist[-2] if len(gr_h2_hist) > 1 else 0.)
        print(f"  [glouton {lev_idx+1}/{params['n_levels_miso']}] miso | "
              f"rang={mask.sum():3d}/{k} | loss_res={loss_total:.6f} "
              f"(data={loss_data:.6f} reg={loss_reg:.6f}) | "
              f"MSE_cum={mse_cum:.6f} | résidu_train={np.mean(gr_res**2):.6f} | "
              f"{times_p1[-1]:.1f}s\n"
              f"      └─ ||f||_H : étage={np.sqrt(h2_step):.4f}  "
              f"CUMULÉ={np.sqrt(h2_cum):.4f}  (||.||²_cum={h2_cum:.4e}, "
              f"Δ||.||²={dh2:+.3e}) | "
              f"loss_H_cum={np.mean(gr_res**2) + lambda_reg*h2_cum:.6f}")

        if last_greedy:                     # seul le dernier étage devient un expert
            F_train[:,slot] = gr_train
            F_test [:,slot] = gr_test
            F_all  [:,slot] = gr_all
            fi_derivs_list.append(gr_derivs)
            print(f"  {expert_names[slot]:7s} | cascade conservée | "
                  f"MSE={mean_squared_error(y_test, gr_test):.6f} | "
                  f"||f||_H={np.sqrt(gr_h2_hist[-1]):.4f}")
            print("      trajectoire ||f_cum||_H : "
                  + " -> ".join(f"{np.sqrt(v):.4f}" for v in gr_h2_hist))
            slot += 1
        continue

    F_train[:,slot] = f_tr
    F_test [:,slot] = f_te
    F_all  [:,slot] = f_al
    fi_derivs_list.append(step_derivs)

    mse_i = mean_squared_error(y_test, f_te)
    print(f"  {expert_names[slot]:7s} | rang={mask.sum():3d}/{k} | "
          f"loss={loss_total:.6f} (data={loss_data:.6f} reg={loss_reg:.6f}) | "
          f"MSE={mse_i:.6f} | σ∈[{sigmas_sel.min():.2f},{sigmas_sel.max():.2f}] | "
          f"{times_p1[-1]:.1f}s")
    slot += 1


# ══════════════════════════════════════════════════════════════════════════════
# AJOUT DES EXPERTS RBF ET RIDGE AVEC DÉRIVÉES SOBOLEV SANS TENSEUR COMPLET
# ══════════════════════════════════════════════════════════════════════════════
from sklearn.preprocessing import PolynomialFeatures

USE_POLY_EXPERT = params["deg_poly_expert"] >= 0
print("\nCalibration expert polynomial..." if USE_POLY_EXPERT
      else "\nExpert polynomial DÉSACTIVÉ (deg_poly_expert < 0)")

# ── Sous-échantillon pour les dérivées (cohérent avec la Gram) ────────────────
rng_H = np.random.default_rng(seed=999)
idx_H = rng_H.choice(N_all, size=min(N_all, params["n_G_H"]), replace=False)
n_H   = len(idx_H)
X_H   = X_all[idx_H]   # (n_H, d)

w1 = params["weights"].get(1, 0.)
w2 = params["weights"].get(2, 0.)
w3 = params["weights"].get(3, 0.)

if USE_POLY_EXPERT:
    # ── Expert polynomial degré deg_poly_expert (dérivées analytiques exactes) ───
    deg_e   = params["deg_poly_expert"]
    d_      = params["n_ambiant"]
    poly_e  = PolynomialFeatures(degree=deg_e, include_bias=False)
    Phi_tr  = poly_e.fit_transform(X_train)
    ridge_e = Ridge(alpha=1e-8)             # petite régularisation numérique du fit
    ridge_e.fit(Phi_tr, y_train)
    powers_e    = poly_e.powers_            # (n_feat, d) : exposants de chaque monôme
    coefs_e     = ridge_e.coef_.copy()
    intercept_e = float(ridge_e.intercept_)

    pred_poly_train = ridge_e.predict(Phi_tr)
    pred_poly_test  = ridge_e.predict(poly_e.transform(X_test))
    #print(f"  Poly{deg_e} expert : MSE={mean_squared_error(y_test, pred_poly_test):.6f}")

    def poly_deriv_eval(X, powers, coefs, dims=()):
        """Évaluation analytique de la dérivée d'ordre len(dims) du polynôme
        f(x) = Σ_j c_j Π_k x_k^{p_jk} par rapport aux dimensions listées dans
        dims (avec multiplicité). Exacte : aucune différence finie."""
        P = powers.astype(np.float64).copy()
        mult = coefs.astype(np.float64).copy()
        for m in dims:
            mult = mult * P[:, m]     # facteur p (p-1) ... et annulation exacte si p=0
            P[:, m] -= 1
        valid = mult != 0
        if not np.any(valid):
            return np.zeros(len(X))
        Xp = X[:, None, :] ** P[None, valid, :]        # (n, n_valid, d)
        return Xp.prod(axis=2) @ mult[valid]

    delta_phi_poly = poly_deriv_eval(X_H, powers_e, coefs_e) + intercept_e

    if w1 or w2 or w3:
        delta_grad_poly = np.stack(
            [poly_deriv_eval(X_H, powers_e, coefs_e, (k_,)) for k_ in range(d_)], axis=1)
    else:
        delta_grad_poly = None

    if w2 or w3:
        delta_hess_poly = np.zeros((n_H, d_, d_))
        for k_ in range(d_):
            for l_ in range(k_, d_):
                v = poly_deriv_eval(X_H, powers_e, coefs_e, (k_, l_))
                delta_hess_poly[:,k_,l_] = v; delta_hess_poly[:,l_,k_] = v
    else:
        delta_hess_poly = None

    if w3:
        delta_third_poly = np.zeros((n_H, d_, d_, d_))
        from itertools import permutations
        for k_ in range(d_):
            for l_ in range(k_, d_):
                for m_ in range(l_, d_):
                    v = poly_deriv_eval(X_H, powers_e, coefs_e, (k_, l_, m_))
                    for a,b,c in set(permutations((k_,l_,m_))):
                        delta_third_poly[:,a,b,c] = v
    else:
        delta_third_poly = None
    norm_poly = 0.
    if w1: norm_poly += w1*np.mean(np.sum(delta_grad_poly**2,  axis=1))
    if w2: norm_poly += w2*np.mean(np.sum(delta_hess_poly**2,  axis=(1,2)))
    if w3: norm_poly += w3*np.mean(np.sum(delta_third_poly**2, axis=(1,2,3)))
    loss_data_p = np.mean((y_train - pred_poly_train)**2)
    print(f"  Poly{deg_e}  | rang={Phi_tr.shape[1]:3d}/{Phi_tr.shape[1]} | "
          f"loss={loss_data_p + params['lambda_reg']*norm_poly:.6f} "
          f"(data={loss_data_p:.6f} reg={params['lambda_reg']*norm_poly:.6f}) | "
          f"MSE={mean_squared_error(y_test, pred_poly_test):.6f}")

    # ── Ajout aux F et fi_derivs ──────────────────────────────────────────────────
    F_train = np.column_stack([F_train, pred_poly_train])
    F_test  = np.column_stack([F_test,  pred_poly_test])

    fi_derivs_list.append({
        'phi': delta_phi_poly,   'grad': delta_grad_poly,
        'hess': delta_hess_poly, 'third': delta_third_poly})

    expert_names += [f"Poly{deg_e}"]

print(f"  H étendu à {len(fi_derivs_list)} fonctions")

# ══════════════════════════════════════════════════════════════════════════════
# PHASE 2 : argmin sur H = <f_1,...,f_n_levels>
# On cherche α* = argmin_{α} ||y - F_train α||² + λ_H α^T G_H α
# où G_H_{ij} = <f_i, f_j>_Sobolev calculé sur n_G_H points
# ══════════════════════════════════════════════════════════════════════════════
print(f"\nPhase 2 : Gram finale sur H ({len(fi_derivs_list)} fonctions, n_G_H={n_H})...")

# idx_H et n_H déjà définis dans le bloc experts
# Gram de H : G_H[i,j] = <f_i, f_j>_Sobolev sur idx_H
n_lev = len(fi_derivs_list)
G_H = np.zeros((n_lev, n_lev))
w0=params["weights"].get(0,0.); w1=params["weights"].get(1,0.)
w2=params["weights"].get(2,0.); w3=params["weights"].get(3,0.)

# phi de chaque f_i sur idx_H : shape (n_H,)
if w0:
    PHI_H = np.stack([
        fi['phi'] if fi['phi'].shape[0] == n_H else fi['phi'][idx_H]
        for fi in fi_derivs_list], axis=1)
    G_H += w0*(PHI_H.T@PHI_H)/n_H

# NOTE : ordres 1 et 2 -> il faut aussi ACCUMULER dans G_H (et pas seulement
# empiler les tenseurs), et gérer le cas où certains experts n'ont pas la dérivée.
if w1:
    idx_with_grad = [i for i, fi in enumerate(fi_derivs_list) if fi['grad'] is not None]
    if idx_with_grad:
        GRAD_H = np.stack([
            (fi_derivs_list[i]['grad'] if fi_derivs_list[i]['grad'].shape[0] == n_H
             else fi_derivs_list[i]['grad'][idx_H])
            for i in idx_with_grad], axis=1)                  # (n_H, m, d)
        G_part = np.einsum('xik,xjk->ij', GRAD_H, GRAD_H, optimize='optimal') * w1 / n_H
        G_H[np.ix_(idx_with_grad, idx_with_grad)] += G_part

if w2:
    idx_with_hess = [i for i, fi in enumerate(fi_derivs_list) if fi['hess'] is not None]
    if idx_with_hess:
        HESS_H = np.stack([
            (fi_derivs_list[i]['hess'] if fi_derivs_list[i]['hess'].shape[0] == n_H
             else fi_derivs_list[i]['hess'][idx_H])
            for i in idx_with_hess], axis=1)                  # (n_H, m, d, d)
        G_part = np.einsum('xikl,xjkl->ij', HESS_H, HESS_H, optimize='optimal') * w2 / n_H
        G_H[np.ix_(idx_with_hess, idx_with_hess)] += G_part

if w3:
    idx_with_third = [i for i, fi in enumerate(fi_derivs_list) if fi['third'] is not None]
    if idx_with_third:
        thirds = []
        for i in idx_with_third:
            t = fi_derivs_list[i]['third']
            thirds.append(t if t.shape[0] == n_H else t[idx_H])
        # THIRD_H : (n_H, n_with_third, d, d, d)
        THIRD_H = np.stack(thirds, axis=1)
        # Gram partielle sur les fonctions qui ont third
        G_partial = np.einsum('xiklm,xjklm->ij', THIRD_H, THIRD_H, optimize='optimal') * w3 / n_H
        # Insérer dans G_H aux bons indices
        for ii, i in enumerate(idx_with_third):
            for jj, j in enumerate(idx_with_third):
                G_H[i, j] += G_partial[ii, jj]

print(f"  G_H calculée ({n_H} pts)")

# ── SÉLECTION DES EXPERTS PAR LOSS ────────────────────────────────────────────
# Loss du problème de Phase 2 restreint au sous-espace de dim 1 engendré par f_i
# (même train, même G_H, même λ_H) :
#   loss_i = min_c (1/n)||y - c f_i||² + λ_H c² G_H[i,i]
# Solution fermée : c_i* = m_i/(q_i + λ_H G_H[i,i]) avec m_i = <f_i,y>/n,
# q_i = ||f_i||²_n, et  loss_i = mean(y²) - m_i²/(q_i + λ_H G_H[i,i]).
# Chaque expert est ainsi optimalement rescalé avant comparaison.
n_loc  = len(y_train)
m_vec  = F_train.T @ y_train / n_loc
q_vec  = np.sum(F_train**2, axis=0) / n_loc
denom  = q_vec + params["lambda_reg"] * np.diag(G_H)
c_opt  = m_vec / denom
expert_losses = np.mean(y_train**2) - m_vec**2 / denom
best_loss   = expert_losses.min()
sel         = expert_losses <= params["k_loss"] * best_loss
sel_idx     = np.where(sel)[0]

print(f"\n  Sélection experts (k_loss={params['k_loss']}, "
      f"seuil={params['k_loss']*best_loss:.3e}) :")
for i in range(n_lev):
    tag = "GARDÉ " if sel[i] else "écarté"
    print(f"    {expert_names[i]:7s} : loss={expert_losses[i]:.3e}  c*={c_opt[i]:+.3f}  [{tag}]")
print(f"  -> {sel.sum()}/{n_lev} experts retenus")

F_train_sel = F_train[:, sel]
F_test_sel  = F_test[:,  sel]
G_H_sel     = G_H[np.ix_(sel_idx, sel_idx)]

# Résolution sur les experts sélectionnés :
# (F^T F / n + λ_H G_H) α = F^T y / n
n_tr = F_train_sel.shape[0]
M_H  = (F_train_sel.T@F_train_sel)/n_tr + params["lambda_reg"]*G_H_sel
rhs_H = (F_train_sel.T@y_train)/n_tr

eigvals_H,eigvecs_H = np.linalg.eigh(M_H)
thresh_H = max(params["lambda_reg"], 1e-10)
mask_H   = eigvals_H > thresh_H
V_H=eigvecs_H[:,mask_H]; S_H=eigvals_H[mask_H]
alpha_sel = V_H@((V_H.T@rhs_H)/S_H)

# alpha complet (0 sur les experts écartés) pour la suite du script
alpha = np.zeros(n_lev); alpha[sel_idx] = alpha_sel

pred_H_train = F_train_sel @ alpha_sel
pred_H_test  = F_test_sel  @ alpha_sel
mse_H = mean_squared_error(y_test, pred_H_test)
mse_H_train = mean_squared_error(y_train, pred_H_train)

print(f"  rang H = {mask_H.sum()}/{sel.sum()} (sur {n_lev} experts)")
print(f"  MSE train = {mse_H_train:.6f}  MSE test = {mse_H:.6f}")
print(f"  alpha = {alpha.round(3)}")


# ══════════════════════════════════════════════════════════════════════════════
# ABLATION : que vaut H SANS l'expert glouton ?
# On rejoue exactement la même Phase 2 (même k_loss, même λ, même solveur) sur
# le sous-ensemble d'experts privé de misoG, et on compare.
# ══════════════════════════════════════════════════════════════════════════════
def solve_H(keep_mask):
    """Phase 2 restreinte aux experts de keep_mask (bool, taille n_lev).
    Refait la sélection k_loss À L'INTÉRIEUR de ce sous-ensemble, puis résout.
    Retourne (mse_test, mse_train, sel_locale, alpha_complet, rang)."""
    kidx = np.where(keep_mask)[0]
    if len(kidx) == 0:
        return np.nan, np.nan, np.zeros(n_lev, bool), np.zeros(n_lev), 0

    # -- sélection k_loss recalculée sur le sous-ensemble --
    losses_k = expert_losses[kidx]                 # loss 1-D : indép. du sous-ensemble
    sel_k    = losses_k <= params["k_loss"] * losses_k.min()
    sidx     = kidx[sel_k]

    Ftr = F_train[:, sidx]; Fte = F_test[:, sidx]
    GH  = G_H[np.ix_(sidx, sidx)]

    n_t = Ftr.shape[0]
    M   = (Ftr.T @ Ftr) / n_t + params["lambda_reg"] * GH
    rhs = (Ftr.T @ y_train) / n_t
    ev, evec = np.linalg.eigh(M)
    mk = ev > max(params["lambda_reg"], 1e-10)
    V = evec[:, mk]; S = ev[mk]
    a_s = V @ ((V.T @ rhs) / S)

    a_full = np.zeros(n_lev); a_full[sidx] = a_s
    sel_full = np.zeros(n_lev, bool); sel_full[sidx] = True
    return (mean_squared_error(y_test,  Fte @ a_s),
            mean_squared_error(y_train, Ftr @ a_s),
            sel_full, a_full, int(mk.sum()))


greedy_idx = [i for i, nm in enumerate(expert_names) if nm.startswith("misoG")]

if MISO_GREEDY and greedy_idx:
    g = greedy_idx[0]
    print("\n" + "="*70)
    print("ABLATION : H avec vs sans l'expert glouton")
    print("="*70)

    mask_with    = np.ones(n_lev, bool)
    mask_without = np.ones(n_lev, bool); mask_without[g] = False

    mse_w,  mse_w_tr,  sel_w,  a_w,  rk_w  = solve_H(mask_with)
    mse_wo, mse_wo_tr, sel_wo, a_wo, rk_wo = solve_H(mask_without)

    gain     = mse_wo - mse_w                       # >0 : le glouton aide
    gain_rel = gain / mse_wo * 100 if mse_wo > 0 else np.nan

    print(f"  H avec  {expert_names[g]:8s} : MSE_test={mse_w:.6f}  "
          f"MSE_train={mse_w_tr:.6f}  rang={rk_w}/{sel_w.sum()}  "
          f"α_glouton={a_w[g]:+.4f}")
    print(f"  H sans  {expert_names[g]:8s} : MSE_test={mse_wo:.6f}  "
          f"MSE_train={mse_wo_tr:.6f}  rang={rk_wo}/{sel_wo.sum()}")
    verdict = ("AIDE" if gain > 0 else "NUIT" if gain < 0 else "neutre")
    print(f"  -> gain = {gain:+.6f} ({gain_rel:+.2f}%)   [{verdict}]")

    # ── indicateurs de contexte : où se situe le glouton ? ────────────────────
    mse_g_solo = mean_squared_error(y_test, F_test[:, g])
    mse_best_other = min(mean_squared_error(y_test, F_test[:, i])
                         for i in range(n_lev) if i != g)
    print(f"\n  {expert_names[g]} seul : MSE={mse_g_solo:.6f} | "
          f"||f||_H={np.sqrt(G_H[g,g]):.4f} | loss={expert_losses[g]:.3e}")
    print(f"  meilleur autre expert seul : MSE={mse_best_other:.6f}")
    print(f"  H complet : MSE={mse_w:.6f}  "
          f"({'MIEUX' if mse_w < min(mse_g_solo, mse_best_other) else 'pas mieux'} "
          f"que tout expert seul -> complémentarité)")

    # ── corrélations du glouton avec les autres (L2 sur train, et Sobolev) ────
    print(f"\n  Redondance de {expert_names[g]} (|corr| ; 1 = redondant) :")
    fg = F_train[:, g]
    for i in range(n_lev):
        if i == g: continue
        num_l2 = abs(fg @ F_train[:, i])
        den_l2 = np.linalg.norm(fg) * np.linalg.norm(F_train[:, i]) + 1e-30
        c_h = abs(G_H[g, i]) / (np.sqrt(G_H[g, g] * G_H[i, i]) + 1e-30)
        print(f"    vs {expert_names[i]:8s} : corr_L2={num_l2/den_l2:.3f}   "
              f"corr_H={c_h:.3f}")


# ══════════════════════════════════════════════════════════════════════════════
# COMPARAISONS
# ══════════════════════════════════════════════════════════════════════════════
from sklearn.kernel_ridge import KernelRidge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import GridSearchCV

print("\nCalibration RBF (baseline de comparaison uniquement, hors H)...")
gamma_grid = np.logspace(-3,2,20)
param_grid = {'alpha':[1e-8,1e-5,1e-3],'gamma':gamma_grid,'kernel':['rbf']}
kr_cv = GridSearchCV(KernelRidge(), param_grid=param_grid, cv=5,
                     scoring='neg_mean_squared_error', n_jobs=-1)
kr_cv.fit(X_train, y_train)
best_gamma = kr_cv.best_params_['gamma']
best_alpha = kr_cv.best_params_['alpha']
best_err_rbf = mean_squared_error(y_test, kr_cv.predict(X_test))
results = kr_cv.cv_results_; order = np.argsort(results['rank_test_score'])
print("\nTop 3 RBF :")
for i in range(3):
    idx = order[i]; g = results['param_gamma'][idx]; a = results['param_alpha'][idx]
    kr = KernelRidge(kernel='rbf', gamma=float(g), alpha=float(a))
    kr.fit(X_train, y_train)
    print(f"  #{i+1} γ={g:.4f} α={a:.0e} TEST={mean_squared_error(y_test, kr.predict(X_test)):.6f}")

poly=PolynomialFeatures(degree=min(params["deg_P"],8),include_bias=False)
ridge_poly=Ridge(alpha=1e-8); ridge_poly.fit(poly.fit_transform(X_train),y_train)
err_poly=mean_squared_error(y_test,ridge_poly.predict(poly.transform(X_test)))

# ══════════════════════════════════════════════════════════════════════════════
# RÉSUMÉ
# ══════════════════════════════════════════════════════════════════════════════
print("\n"+"="*70); print("RÉSUMÉ"); print("="*70)
print(f"Polynomial Ridge  : {err_poly:.6f}")
print(f"RBF (γ={best_gamma:.4f})  : {best_err_rbf:.6f}")
print(f"\nMSE individuelles des experts :")
for i in range(n_lev):
    mse_i = mean_squared_error(y_test, F_test[:,i])
    tag = "" if sel[i] else "  (écarté)"
    print(f"  {expert_names[i]:7s} : {mse_i:.6f}  loss={expert_losses[i]:.3e}{tag}")
print(f"\nSobolev sous-espace H : {mse_H:.6f}  (rang={mask_H.sum()})")
print(f"\nweights={params['weights']}  λ={params['lambda_reg']} (unique)")
print(f"n_levels={params['n_levels']} (+{params['n_levels_miso']} miso"
      f"{' glouton->1' if MISO_GREEDY else ''}, "
      f"+{params['n_levels_wnd']} wnd, "
      f"poly={'OFF' if not USE_POLY_EXPERT else params['deg_poly_expert']})  "
      f"n_centres={params['n_centres']}  n_dict={params['n_dict']}  "
      f"k_loss={params['k_loss']} ({sel.sum()}/{n_lev} experts retenus)")
print(f"n_G={params['n_G']}  n_G_H={params['n_G_H']}")
print(f"σ∈[{params['sigma_min']},{params['sigma_max']}]")
print(f"temps phase 1 : {sum(times_p1):.1f}s total  ({np.mean(times_p1):.1f}s/niveau)")

from sklearn.metrics import roc_auc_score, accuracy_score

seuil = np.median(y_test)
y_test_class  = (y_test  > seuil).astype(int)
y_train_class = (y_train > seuil).astype(int)

pred_H     = F_test  @ alpha
pred_rbf = kr_cv.predict(X_test)
pred_ridge = ridge_poly.predict(poly.transform(X_test))

print("\n" + "="*70)
print("CLASSIFICATION (seuil = médiane de y_test)")
print("="*70)
for nom, pred in [("Sobolev H", pred_H), ("RBF", pred_rbf), ("Ridge poly", pred_ridge)]:
    pred_class = (pred > seuil).astype(int)
    acc = accuracy_score(y_test_class, pred_class)
    try:
        auc = roc_auc_score(y_test_class, pred)
    except Exception:
        auc = float('nan')
    print(f"  {nom:12s} : accuracy={acc:.4f}  AUC={auc:.4f}")

Génération du cloud sur {Q=0}...
  collectés:89/50 (44.5%)
  collectés:8630/5000 (43.1%)
